# Chapter 2: CSS Selectors as a Query Language

CSS selectors are the primary tool for targeting elements in HTML. BeautifulSoup's `select()` method
accepts any CSS selector and returns matching elements. Knowing the full CSS selector vocabulary
lets you precisely target any element on any page.

This notebook is a practical reference. All examples run against the ShopSphere SSR demo site.

In [9]:
import httpx
from bs4 import BeautifulSoup

def fetch(url):
    return BeautifulSoup(httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}).text, "lxml")

# Load ShopSphere listing page and a product detail page
listing = fetch("http://localhost:8001/products")
product = fetch("http://localhost:8001/products/macbook-pro-14")

print("Pages loaded")

Pages loaded


## 2.1 Basic Selectors

In [10]:
# Element type
print("h2 elements:", len(listing.select("h2")))

# Class
print("Elements with class 'product-card':", len(listing.select(".product-card")))

# ID
print("Element with id 'app':", len(listing.select("#app")))

# Type + class (AND)
print("article.product-card:", len(listing.select("article.product-card")))

# Multiple classes (AND)
print(".product-category-badge.large:", len(product.select(".product-category-badge.large")))

h2 elements: 10
Elements with class 'product-card': 10
Element with id 'app': 0
article.product-card: 10
.product-category-badge.large: 1


## 2.2 Descendant and Child Selectors

In [11]:
# Descendant (any depth)
titles = listing.select(".product-card h2")
print(f"Product titles (descendant): {[t.get_text(strip=True)[:30] for t in titles[:3]]}")

# Direct child only (>)
direct = listing.select(".product-card > .product-card-body")
print(f"Direct child .product-card-body: {len(direct)}")

# Links inside product cards
links = listing.select(".product-card a.product-link")
print(f"Product links: {len(links)}")
print(f"First href: {links[0]['href'] if links else None}")

Product titles (descendant): ['MacBook Pro 14"', 'Dell XPS 15', 'Lenovo ThinkPad X1 Carbon Gen ']
Direct child .product-card-body: 10
Product links: 10
First href: /products/macbook-pro-14


## 2.3 Attribute Selectors

In [12]:
# Has attribute
data_rating = listing.select("[data-rating]")
print(f"Elements with data-rating: {len(data_rating)}")

# Attribute equals
specific = product.select("[data-rating='4.8']")
print(f"Rating exactly 4.8: {len(specific)}")

# Attribute contains
contains = product.select("[class*='rating']")
print(f"Classes containing 'rating': {[el['class'] for el in contains]}")

# Attribute starts with
starts = product.select("[data-product-id^='PROD']")
print(f"data-product-id starts with PROD: {len(starts)}")

Elements with data-rating: 10
Rating exactly 4.8: 1
Classes containing 'rating': [['product-rating-row'], ['rating-value'], ['rating-label']]
data-product-id starts with PROD: 1


## 2.4 Pseudo-classes for Structural Selection

In [13]:
# Specs table selectors
spec_table = product.select_one("table.specs-table")
if spec_table:
    # First row
    first_row = spec_table.select_one("tr:first-child")
    print(f"First spec row: {first_row.get_text(strip=True) if first_row else None}")
    
    # Last row
    last_row = spec_table.select_one("tr:last-child")
    print(f"Last spec row: {last_row.get_text(strip=True) if last_row else None}")
    
    # Nth row (1-indexed)
    third_row = spec_table.select_one("tr:nth-child(3)")
    print(f"Third spec row: {third_row.get_text(strip=True) if third_row else None}")
    
    # Specific columns
    keys = spec_table.select("td:nth-child(1)")
    vals = spec_table.select("td:nth-child(2)")
    print(f"Spec keys: {[k.get_text(strip=True) for k in keys]}")

First spec row: ProcessorApple M3 Pro
Last spec row: BatteryUp to 18 hours
Third spec row: Storage512GB SSD
Spec keys: ['Processor', 'RAM', 'Storage', 'Display', 'Battery']


## 2.5 The :contains() Pseudo-selector (BeautifulSoup extension)

In [14]:
# BeautifulSoup extends CSS with :contains() for text matching
# Note: this is not standard CSS - it's a BS4 extension
import re

def select_contains(soup, selector, text):
    """Emulate :contains() - select elements matching selector AND containing text."""
    elements = soup.select(selector)
    return [el for el in elements if text.lower() in el.get_text().lower()]

# Find the 'In Stock' status element
# Detail page uses .stock-status (listing cards use .product-stock)
in_stock = select_contains(product, ".stock-status", "In Stock")
print(f"In Stock elements: {len(in_stock)}")
if in_stock:
    print(f"  Text: {in_stock[0].get_text(strip=True)!r}")

# Find badges with a specific text
laptops_badge = select_contains(listing, ".product-category-badge", "Laptops")
print(f"\nLaptop category badges: {len(laptops_badge)}")

In Stock elements: 1
  Text: 'In Stock'

Laptop category badges: 4


## 2.6 Combining Selectors

In [15]:
# Comma-separated: match either selector
all_links = listing.select("a.product-link, a.pagination-btn")
print(f"Product links OR pagination links: {len(all_links)}")

# Build a full product extraction using combined selectors
cards = listing.select("article.product-card")
products = []
for card in cards:
    products.append({
        "name": card.select_one("h2.product-name a").get_text(strip=True),
        "url": card.select_one("a.product-link")["href"],
        "price": card.select_one(".product-price").get_text(strip=True),
        "rating": card.select_one(".product-rating")["data-rating"],
        "category": card.select_one(".product-category-badge").get_text(strip=True),
        "in_stock": "in-stock" in card.select_one(".product-stock")["class"],
    })

for p in products[:3]:
    print(p)

Product links OR pagination links: 16
{'name': 'MacBook Pro 14"', 'url': '/products/macbook-pro-14', 'price': '$1999.00', 'rating': '4.8', 'category': 'Laptops', 'in_stock': True}
{'name': 'Dell XPS 15', 'url': '/products/dell-xps-15', 'price': '$1499.00', 'rating': '4.6', 'category': 'Laptops', 'in_stock': True}
{'name': 'Lenovo ThinkPad X1 Carbon Gen 11', 'url': '/products/lenovo-thinkpad-x1-carbon', 'price': '$1299.00', 'rating': '4.7', 'category': 'Laptops', 'in_stock': True}


## 2.7 CSS Selector Quick Reference

| Selector | Matches | Example |
|----------|---------|----------|
| `div` | All `<div>` elements | `select('div')` |
| `.price` | Elements with class `price` | `select('.price')` |
| `#main` | Element with id `main` | `select('#main')` |
| `span.price` | `<span>` with class `price` | `select('span.price')` |
| `.a.b` | Elements with BOTH classes | `select('.card.active')` |
| `div p` | `<p>` anywhere inside `<div>` | `select('div p')` |
| `div > p` | `<p>` that is DIRECT child of `<div>` | `select('div > p')` |
| `[href]` | Any element with href attribute | `select('[href]')` |
| `[href*="jobs"]` | href contains "jobs" | `select('[href*="jobs"]')` |
| `a, button` | Either `<a>` or `<button>` | `select('a, button')` |
| `li:first-child` | First `<li>` in its parent | `select('li:first-child')` |
| `td:nth-child(2)` | Second `<td>` in its parent row | `select('td:nth-child(2)')` |